In [ ]:
import os
import time
import subprocess
import chromadb
from google import genai
from google.genai import types
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

# ---------------------------------------------------------
# 1. Configuration & Setup
# ---------------------------------------------------------
# Replace with your actual Google API key
os.environ["GEMINI_API_KEY"] = "" # input your API key here
client = genai.Client()

# Initialize ChromaDB
DB_PATH = "./cctv_chroma_db"
chroma_client = chromadb.PersistentClient(path=DB_PATH)
# We use a new collection name to avoid mixing with any old text-based vectors
collection = chroma_client.get_or_create_collection(name="direct_video_vectors")
dimension = 768

# ---------------------------------------------------------
# 2. FFmpeg Video Utilities
# ---------------------------------------------------------
def get_video_duration(video_path):
    """Uses FFprobe to get the exact duration of the raw video."""
    cmd = ["ffprobe", "-v", "error", "-show_entries", "format=duration", 
           "-of", "default=noprint_wrappers=1:nokey=1", video_path]
    return float(subprocess.check_output(cmd).decode('utf-8').strip())

def extract_video_clip(input_path, start_sec, duration, output_path):
    """Losslessly slices a specific time window from the raw video."""
    cmd = [
        "ffmpeg", "-y", "-ss", str(start_sec), "-i", input_path, 
        "-t", str(duration), "-c", "copy", "-avoid_negative_ts", "make_zero", output_path
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return output_path if os.path.exists(output_path) else None

# ---------------------------------------------------------
# 3. PHASE 1: Direct Multimodal Ingestion (Run Once)
# ---------------------------------------------------------

def ingest_raw_video_direct(video_path, chunk_duration=15.0):
    """
    Directly embeds raw video chunks into ChromaDB using INLINE DATA,
    bypassing the Google Cloud File API entirely.
    """
    video_id = os.path.splitext(os.path.basename(video_path))[0]
    duration = get_video_duration(video_path)
    
    print(f"\n=== 🎬 STARTING DIRECT MULTIMODAL INGESTION: {video_id} ({duration:.1f}s) ===")
    
    current_sec = 0.0
    chunk_idx = 0
    
    while current_sec < duration:
        end_sec = min(current_sec + chunk_duration, duration)
        clip_path = f"temp_ingest_{chunk_idx}.mp4"
        
        # 1. Extract 15-second Chunk
        extract_video_clip(video_path, current_sec, chunk_duration, clip_path)
        print(f"\n⚙️ Processing Chunk {chunk_idx} [{current_sec:.1f}s - {end_sec:.1f}s]...")
        
        try:
            # 2. READ VIDEO AS RAW BYTES (Bypasses the buggy File API)
            with open(clip_path, "rb") as f:
                video_bytes = f.read()
            
            # 3. DIRECT EMBEDDING (Inline Bytes -> Math)
            embed_response = client.models.embed_content(
                model='gemini-embedding-2-preview',
                # Pass the raw bytes directly to the model
                contents=types.Part.from_bytes(data=video_bytes, mime_type="video/mp4"),
                config=types.EmbedContentConfig(
                    task_type="RETRIEVAL_DOCUMENT",
                    output_dimensionality=dimension
                )
            )
            vector = embed_response.embeddings[0].values
            
            # 4. Save Vector to ChromaDB 
            collection.add(
                embeddings=[vector],
                documents=[""], # No text!
                metadatas=[{"video_id": video_id, "start_sec": current_sec, "end_sec": end_sec}],
                ids=[f"{video_id}_chunk_{chunk_idx}"]
            )
            print("   ✅ Video chunk embedded and saved.")
            
        except Exception as e:
            print(f"   ❌ Error processing chunk {chunk_idx}: {e}")
            
        # Clean up local file
        if os.path.exists(clip_path):
            os.remove(clip_path)
            
        current_sec += chunk_duration
        chunk_idx += 1
        
    print(f"=== 🎉 DIRECT INGESTION COMPLETE! '{video_id}' is now searchable. ===")
# ---------------------------------------------------------
# 4. PHASE 2: Chat & Retrieval (Run Anytime)
# ---------------------------------------------------------


# ---------------------------------------------------------
# 4. PHASE 2: Chat & Retrieval (LLM CONTEXT PRUNING)
# ---------------------------------------------------------
def chat_with_raw_video_direct(user_query, raw_video_path, chat_history=None):
    """
    Retrieves top matches, groups them into isolated chronological clusters, 
    extracts clips, and synthesizes an answer using dynamic LLM memory management.
    """
    if chat_history is None:
        chat_history = []

    video_id = os.path.splitext(os.path.basename(raw_video_path))[0]
    print(f"\n=== 🕵️‍♂️ CCTV ASSISTANT QUERY ===\nQuestion: '{user_query}'")

    # --- Step 1: LLM-Driven Conversational Query Rewriting (CQR) ---
    print("🦙 Asking Gemma 4 to resolve context and rewrite the query...")
    
    # Format the entire history into a readable string
    full_history_text = ""
    for idx, turn in enumerate(chat_history):
        full_history_text += f"[Turn {idx+1}] User: {turn['question']} | AI: {turn['answer']}\n"
    
    system_prompt = f"""You are an elite intent parser for a surveillance system.
Your job is to read the user's CURRENT QUERY and the FULL CHAT HISTORY, and output two things separated by a '|' character:
1. RELEVANT HISTORY: A brief summary of ONLY the facts from the history needed to understand the current query. If no history is needed, write 'NONE'.
2. REFINED QUERY: Rewrite the user's current query into a single, fully self-contained natural language sentence. Resolve any pronouns (e.g., replace "he" with "the man in the red shirt") and include necessary context from previous turns so the query makes perfect sense on its own. DO NOT output comma-separated keywords. Write it exactly as a human would ask a complete, standalone question.

FULL CHAT HISTORY:
{full_history_text if full_history_text else "No previous history."}

CURRENT QUERY: {user_query}

Respond strictly in this format: RELEVANT HISTORY | REFINED QUERY"""

    try:
        #local_llm = ChatOllama(model="llama3")
        #ai_message = local_llm.invoke([HumanMessage(content=system_prompt)])

        ai_message = client.models.generate_content(
            model='gemma-4-9b-it', # Ensure you use the '-it' (Instruction Tuned) version
            contents=system_prompt
        )
        llm_output = ai_message.content.strip()


        
        # Robust parsing: Ensure the LLM actually used the '|' delimiter
        if '|' in llm_output:
            extracted_memory, refined_query = [part.strip() for part in llm_output.split('|', 1)]
        else:
            print("⚠️ LLaMA 3 formatting error. Bypassing extraction.")
            extracted_memory = "NONE"
            refined_query = user_query
            
        print(f"🧠 Pruned Memory: {extracted_memory}")
        print(f"✨ Refined Query: {refined_query}")
        
    except Exception as e:
        print(f"⚠️ Local LangChain query rewriting failed: {e}")
        extracted_memory = "NONE"
        refined_query = user_query
    
    # --- Step 2: Embed the Refined Natural Language Query ---
    print("🔢 Embedding refined natural language query...")
    query_embed = client.models.embed_content(
        model='gemini-embedding-2-preview',
        contents=refined_query, # Now using the fully constructed sentence!
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_QUERY",
            output_dimensionality=dimension
        )
    )
    user_vector = query_embed.embeddings[0].values
    
    # --- Step 3: Search ChromaDB ---
    print("🔍 Searching Vector Database for top matches...")
    results = collection.query(
        query_embeddings=[user_vector],
        n_results=5, 
        where={"video_id": video_id}
    )
    
    if not results['metadatas'] or not results['metadatas'][0]:
        final_answer = "I could not find any events matching that description."
        chat_history.append({"question": user_query, "answer": final_answer})
        return final_answer, chat_history
        
    # --- Step 4: Multi-Cluster Logic ---
    chunks = results['metadatas'][0]
    chunks_sorted = sorted(chunks, key=lambda x: x['start_sec'])
    
    clusters = []
    current_cluster = [chunks_sorted[0]]
    
    for chunk in chunks_sorted[1:]:
        last_chunk_end = current_cluster[-1]['end_sec']
        if chunk['start_sec'] - last_chunk_end <= 45.0:
            current_cluster.append(chunk)
        else:
            clusters.append(current_cluster)
            current_cluster = [chunk]
            
    clusters.append(current_cluster)
    print(f"🧩 System identified {len(clusters)} distinct time event(s) across the video.")

    # --- Step 5: Dynamic Trimming & Inline Loading ---
    video_parts = []
    local_clip_paths = []
    
    try:
        for i, cluster in enumerate(clusters):
            cluster_start = min([m['start_sec'] for m in cluster])
            cluster_end = max([m['end_sec'] for m in cluster])
            
            safe_start = max(0.0, cluster_start - 3.0)
            clip_duration = (cluster_end - safe_start) + 3.0 
            
            clip_path = f"temp_cluster_{i}_{int(time.time())}.mp4"
            local_clip_paths.append(clip_path)
            
            print(f"✂️ Extracting Clip {i+1}/{len(clusters)}: {safe_start:.1f}s to {safe_start+clip_duration:.1f}s...")
            extract_video_clip(raw_video_path, safe_start, clip_duration, clip_path)
            
            print(f"📦 Loading Clip {i+1} inline to bypass Cloud Storage...")
            with open(clip_path, "rb") as f:
                video_bytes = f.read()
                
            video_parts.append(
                types.Part.from_bytes(data=video_bytes, mime_type="video/mp4")
            )

        # --- Step 6: Synthesis Continuity (Using Pruned Memory) ---
        print("🧠 Asking Gemini 2.5 Flash to synthesize an answer...")
        
        system_prompt = f"""You are an elite CCTV analysis AI. 
You are being provided with {len(video_parts)} chronologically ordered video clips.
Watch all the clips closely to answer the CURRENT USER QUERY.

INVESTIGATION CONTEXT (Extracted from past turns): 
{extracted_memory}

Use the Investigation Context to resolve any pronouns or track ongoing subjects. 
Return when the event happens, which clip it happens in, and a detailed description."""

        api_contents = video_parts + [system_prompt, f"CURRENT USER QUERY: {user_query}"]

        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=api_contents
        )
        final_answer = response.text

    except Exception as e:
        final_answer = f"Error during final visual analysis: {e}"
        
    finally:
        print("🧹 Cleaning up local temporary files...")
        for l_file in local_clip_paths:
            if os.path.exists(l_file):
                os.remove(l_file)
    
    # Save this turn into the session history before returning
    chat_history.append({"question": user_query, "answer": final_answer})
    return final_answer, chat_history



In [11]:

# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Anomaly-Videos-Part-1/Abuse/Abuse038_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
#ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=15.0)

# --- STEP 2: CHAT ---
QUESTION = "Is there any dangerous activity?"

answer = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)


=== 🕵️‍♂️ CCTV ASSISTANT QUERY ===
Question: 'Is there any dangerous activity?'
🦙 Asking Local LLaMA 3 (via LangChain) to translate human intent...
👁️ Visual Search Terms: person, individual, human, actor, trespasser, violator, miscreant, malefactor; aggressive behavior, violent act, harmful deed, risky move, hazardous occurrence, unsafe situation, perilous event
🔢 Embedding text query...
🔍 Searching Vector Database for top 3 matches...
🧩 System identified 1 distinct time event(s) across the video.
✂️ Extracting Clip 1/1: 0.0s to 32.7s...
📦 Loading Clip 1 inline to bypass Cloud Storage...
🧠 Asking Gemini 2.5 Flash to synthesize an answer across all clips...
🧹 Cleaning up local temporary files...

--- 🤖 FINAL ANSWER ---
Yes, there is dangerous activity in the video.

**Time of event:** 00:05 - 00:17 (Clip 1)
**Description:** Multiple dogs are seen repeatedly running across a busy road with active vehicle traffic. This poses a significant danger to the animals themselves and could poten

In [ ]:
QUESTION = "Is there any person injured in the incident?"

answer = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:

# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Anomaly-Videos-Part-1/Abuse/Abuse042_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=30.0)

# --- STEP 2: CHAT ---
QUESTION = "Did the kid get abused?"

answer = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Anomaly-Videos-Part-1/Abuse/Abuse042_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
#ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=30.0)

# --- STEP 2: CHAT ---
QUESTION = "Did the woman used phone when interact with the kid?"

answer = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Anomaly-Videos-Part-1/Abuse/Abuse042_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
#ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=30.0)

# --- STEP 2: CHAT ---
QUESTION = "did the woman make a phone call when interacting with the child?"

answer = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Anomaly-Videos-Part-1/Abuse/Abuse042_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
#ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=30.0)

# --- STEP 2: CHAT ---
QUESTION = "Did the kid be left unattended?"

answer = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Anomaly-Videos-Part-1/Abuse/Abuse042_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
#ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=30.0)

# --- STEP 2: CHAT ---
QUESTION = "Did the kid be left unattended, can you give me the second in the video the kid be left unattended?"

answer = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Anomaly-Videos-Part-1/Abuse/Abuse039_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=30.0)

# --- STEP 2: CHAT ---
QUESTION = "What happend in the video?"

answer = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Normal_Videos_for_Event_Recognition/Normal_Videos_576_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=30.0)

# --- STEP 2: CHAT ---
QUESTION = "Does any approach the red Isuzu car?"

answer = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [16]:
# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Normal_Videos_for_Event_Recognition/Normal_Videos_576_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
#ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=30.0) ## do the ingestion overlapping

# --- STEP 2: CHAT ---
QUESTION = "Did any yellow van or yello truck appear in the video?"

answer = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)


=== 🕵️‍♂️ CCTV ASSISTANT QUERY ===
Question: 'Did any yellow van or yello truck appear in the video?'
🦙 Asking Local LLaMA 3 (via LangChain) to translate human intent...
👁️ Visual Search Terms: yellow vehicle, yellow van, yellow truck, gold-colored auto, golden lorry, canary-yellow motorized transport
🔢 Embedding text query...
🔍 Searching Vector Database for top 3 matches...
🧩 System identified 3 distinct time event(s) across the video.
✂️ Extracting Clip 1/3: 27.0s to 63.0s...
📦 Loading Clip 1 inline to bypass Cloud Storage...
✂️ Extracting Clip 2/3: 117.0s to 153.0s...
📦 Loading Clip 2 inline to bypass Cloud Storage...
✂️ Extracting Clip 3/3: 237.0s to 363.0s...
📦 Loading Clip 3 inline to bypass Cloud Storage...
🧠 Asking Gemini 2.5 Flash to synthesize an answer across all clips...
🧹 Cleaning up local temporary files...

--- 🤖 FINAL ANSWER ---
Yes, a yellow truck appeared in the video.

The yellow truck can be seen in the **third clip**, from **01:55 to 01:57**.


In [13]:
# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Anomaly-Videos-Part-1/Arrest/Arrest040_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=30.0) ## do the ingestion overlapping

# --- STEP 2: CHAT ---
QUESTION = "Any dangerous activities detected?"

answer = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)


=== 🎬 STARTING DIRECT MULTIMODAL INGESTION: Arrest040_x264 (59.1s) ===

⚙️ Processing Chunk 0 [0.0s - 30.0s]...
   ✅ Video chunk embedded and saved.

⚙️ Processing Chunk 1 [30.0s - 59.1s]...
   ✅ Video chunk embedded and saved.
=== 🎉 DIRECT INGESTION COMPLETE! 'Arrest040_x264' is now searchable. ===

=== 🕵️‍♂️ CCTV ASSISTANT QUERY ===
Question: 'Any dangerous activities detected?'
🦙 Asking Local LLaMA 3 (via LangChain) to translate human intent...
👁️ Visual Search Terms: person attempting to break window, attempted vandalism, suspicious behavior, individual trying to force entry, potential trespassing, alarming movement, concerning activity
🔢 Embedding text query...
🔍 Searching Vector Database for top 3 matches...
🧩 System identified 1 distinct time event(s) across the video.
✂️ Extracting Clip 1/1: 0.0s to 62.1s...
📦 Loading Clip 1 inline to bypass Cloud Storage...
🧠 Asking Gemini 2.5 Flash to synthesize an answer across all clips...
🧹 Cleaning up local temporary files...

--- 🤖 FINA

In [25]:
# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Anomaly-Videos-Part-1/Arson/Arson002_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
#ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=45.0) ## do the ingestion overlapping

# --- STEP 2: CHAT ---
QUESTION = "What cause the fire?"

answer = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)


=== 🕵️‍♂️ CCTV ASSISTANT QUERY ===
Question: 'What cause the fire?'
🦙 Asking Local LLaMA 3 to analyze history and expand conceptual intent...
🧠 Pruned Memory: NONE
✨ Expanded Query: investigation, inquiry, origin, source, reason, explanation
🔢 Embedding conceptual search query...
🔍 Searching Vector Database for top matches...
🧩 System identified 1 distinct time event(s) across the video.
✂️ Extracting Clip 1/1: 0.0s to 151.0s...
📦 Loading Clip 1 inline to bypass Cloud Storage...
🧠 Asking Gemini 2.5 Flash to synthesize an answer...
🧹 Cleaning up local temporary files...

--- 🤖 FINAL ANSWER ---
('The person in the video poured liquid from a can onto the ground near the wall at approximately 00:30-00:59, then bent down with a lighter in their hand, and ignited the liquid at 01:34, causing the fire.', [{'question': 'What cause the fire?', 'answer': 'The person in the video poured liquid from a can onto the ground near the wall at approximately 00:30-00:59, then bent down with a lighter in

In [ ]:
# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Anomaly-Videos-Part-1/Arson/Arson002_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
#ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=45.0) ## do the ingestion overlapping

# --- STEP 2: CHAT ---
QUESTION = "Did he comeback after the fire stop?"

answer = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)


=== 🕵️‍♂️ CCTV ASSISTANT QUERY ===
Question: 'Did he comeback after the fire stop?'
🦙 Asking Local LLaMA 3 to analyze history and expand conceptual intent...
🧠 Pruned Memory: NONE
✨ Expanded Query: fire, incident, emergency, disaster, return, comeback
🔢 Embedding conceptual search query...
🔍 Searching Vector Database for top matches...
🧩 System identified 1 distinct time event(s) across the video.
✂️ Extracting Clip 1/1: 0.0s to 151.0s...
📦 Loading Clip 1 inline to bypass Cloud Storage...
🧠 Asking Gemini 2.5 Flash to synthesize an answer...
🧹 Cleaning up local temporary files...

--- 🤖 FINAL ANSWER ---
('No, he did not come back after the fire stopped. The person is last seen at 01:34 in Clip 1, falling to the ground and then quickly moving out of frame up the stairs as the fire erupts. The fire then dies down, but the person does not reappear in the remaining footage.', [{'question': 'Did he comeback after the fire stop?', 'answer': 'No, he did not come back after the fire stopped. T

In [36]:
# Point this to your completely raw, unannotated video
session_history = []
TARGET_VIDEO = "Anomaly-Videos-Part-1/Assault/Assault002_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
#ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=45.0) ## do the ingestion overlapping

# --- STEP 2: CHAT ---
QUESTION = "What items are used to attack the victim?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)


=== 🕵️‍♂️ CCTV ASSISTANT QUERY ===
Question: 'What items are used to attack the victim?'
🦙 Asking Local LLaMA 3 to resolve context and rewrite the query...
🧠 Pruned Memory: NONE
✨ Refined Query: Who is the victim and what items are used to attack them?
🔢 Embedding refined natural language query...
🔍 Searching Vector Database for top matches...
🧩 System identified 1 distinct time event(s) across the video.
✂️ Extracting Clip 1/1: 0.0s to 87.1s...
📦 Loading Clip 1 inline to bypass Cloud Storage...
🧠 Asking Gemini 2.5 Flash to synthesize an answer...
🧹 Cleaning up local temporary files...

--- 🤖 FINAL ANSWER ---
In the video, the following item is used to attack people:

*   **A chair:** At 00:13, a person in a white shirt picks up a chair and throws it into the crowd where the altercation is happening.


In [37]:
QUESTION = "So how many of them in total?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)


=== 🕵️‍♂️ CCTV ASSISTANT QUERY ===
Question: 'So how many of them in total?'
🦙 Asking Local LLaMA 3 to resolve context and rewrite the query...
🧠 Pruned Memory: NONE
✨ Refined Query: A total number of people who used chairs as weapons is needed.
🔢 Embedding refined natural language query...
🔍 Searching Vector Database for top matches...
🧩 System identified 1 distinct time event(s) across the video.
✂️ Extracting Clip 1/1: 0.0s to 87.1s...
📦 Loading Clip 1 inline to bypass Cloud Storage...
🧠 Asking Gemini 2.5 Flash to synthesize an answer...
🧹 Cleaning up local temporary files...

--- 🤖 FINAL ANSWER ---
The maximum number of people visible in the room at any given time is estimated to be around **26-28 individuals**. This peak occurs between approximately **00:35 and 00:40** in the video, when the room is most densely populated.

It is challenging to provide an exact count of every unique individual who appears throughout the entire video due to continuous movement, people overlapping,

In [40]:
QUESTION = "Is there any other weapon apart from the one you just mention?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)


=== 🕵️‍♂️ CCTV ASSISTANT QUERY ===
Question: 'Is there any other weapon apart from the one you just mention?'
🦙 Asking Local LLaMA 3 to resolve context and rewrite the query...
🧠 Pruned Memory: NONE
✨ Refined Query: Is there another object used to attack someone, excluding the chair that was thrown into the crowd?
🔢 Embedding refined natural language query...
🔍 Searching Vector Database for top matches...
🧩 System identified 1 distinct time event(s) across the video.
✂️ Extracting Clip 1/1: 0.0s to 87.1s...
📦 Loading Clip 1 inline to bypass Cloud Storage...
🧠 Asking Gemini 2.5 Flash to synthesize an answer...
🧹 Cleaning up local temporary files...

--- 🤖 FINAL ANSWER ---
Based on the video provided, I can see a couple of objects being used as weapons:

1.  **0:02 - Clip 1:** A person wearing a yellow top picks up a wooden stick or bat from the floor and starts using it to hit other people. This is clearly visible as a weapon.
2.  **0:01 - Clip 1:** Before that, at 0:01, the person in 

In [23]:
#Normal_Videos_439_x264
# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Normal_Videos_for_Event_Recognition/Normal_Videos_439_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
#ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=45) ## do the ingestion overlapping

# --- STEP 2: CHAT ---
QUESTION = "Did anyone steal anything from the store?"

answer = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)


=== 🕵️‍♂️ CCTV ASSISTANT QUERY ===
Question: 'Did anyone steal anything from the store?'
🦙 Asking Local LLaMA 3 (via LangChain) to translate human intent...
👁️ Visual Search Terms: person, theft, shoplifting, taking, removing, grabbing, pilfering, swiping, snatching, lifting, purloining, appropriating, misappropriating
🔢 Embedding text query...
🔍 Searching Vector Database for top 3 matches...
🧩 System identified 1 distinct time event(s) across the video.
✂️ Extracting Clip 1/1: 0.0s to 143.9s...
📦 Loading Clip 1 inline to bypass Cloud Storage...
🧠 Asking Gemini 2.5 Flash to synthesize an answer across all clips...
🧹 Cleaning up local temporary files...

--- 🤖 FINAL ANSWER ---
No, based on the provided video, no one stole anything from the store. All customers observed making purchases at the counter appeared to complete their transactions.


In [29]:
session_history = []

TARGET_VIDEO = "Anomaly-Videos-Part-1/Arson/Arson002_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
#ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=45) ## do the ingestion overlapping

# --- STEP 2: CHAT ---
QUESTION = "What cause the fire?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

QUESTION = "What color was their shirt?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)


=== 🕵️‍♂️ CCTV ASSISTANT QUERY ===
Question: 'What cause the fire?'
🦙 Asking Local LLaMA 3 to analyze history and expand conceptual intent...
🧠 Pruned Memory: NONE
✨ Expanded Query: cause, reason, explanation, incident, event
🔢 Embedding conceptual search query...
🔍 Searching Vector Database for top matches...
🧩 System identified 1 distinct time event(s) across the video.
✂️ Extracting Clip 1/1: 0.0s to 151.0s...
📦 Loading Clip 1 inline to bypass Cloud Storage...
🧠 Asking Gemini 2.5 Flash to synthesize an answer...
🧹 Cleaning up local temporary files...

--- 🤖 FINAL ANSWER ---
The fire was caused by the individual pouring an accelerant from a container onto the ground near the wall and then igniting it. This happens at 00:01:34 in the first and only clip.

=== 🕵️‍♂️ CCTV ASSISTANT QUERY ===
Question: 'What color was their shirt?'
🦙 Asking Local LLaMA 3 to analyze history and expand conceptual intent...
🧠 Pruned Memory: NONE
✨ Expanded Query: clothing, attire, garment, apparel
🔢 Embedd

In [ ]:
session_history = []

TARGET_VIDEO = "Anomaly-Videos-Part-1/Assault/Assault028_x264.mp4" # at night with snow

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=45) ## do the ingestion overlapping

# --- STEP 2: CHAT ---
QUESTION = "Any dangerous activity in the video?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

QUESTION = "How many people involve?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

QUESTION = "Does he still alive?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

print("session")
print(session_history)


=== 🎬 STARTING DIRECT MULTIMODAL INGESTION: Assault028_x264 (87.8s) ===

⚙️ Processing Chunk 0 [0.0s - 45.0s]...
   ✅ Video chunk embedded and saved.

⚙️ Processing Chunk 1 [45.0s - 87.8s]...
   ✅ Video chunk embedded and saved.
=== 🎉 DIRECT INGESTION COMPLETE! 'Assault028_x264' is now searchable. ===

=== 🕵️‍♂️ CCTV ASSISTANT QUERY ===
Question: 'Any dangerous activity in the video?'
🦙 Asking Local LLaMA 3 to analyze history and expand conceptual intent...
🧠 Pruned Memory: NONE
✨ Expanded Query: danger, threat, security breach, risk, alarm
🔢 Embedding conceptual search query...
🔍 Searching Vector Database for top matches...
🧩 System identified 1 distinct time event(s) across the video.
✂️ Extracting Clip 1/1: 0.0s to 90.8s...
📦 Loading Clip 1 inline to bypass Cloud Storage...
🧠 Asking Gemini 2.5 Flash to synthesize an answer...
🧹 Cleaning up local temporary files...

--- 🤖 FINAL ANSWER ---
Yes, there is dangerous activity in the video.

**When it happens:** From 00:23 to 00:40 in the

In [52]:
session_history = []

TARGET_VIDEO = "Anomaly-Videos-Part-1/Assault/Assault028_x264.mp4" # at night with snow

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
#ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=45) ## do the ingestion overlapping

# --- STEP 2: CHAT ---
QUESTION = "What is the color of the car the attackers use to run away?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)


=== 🕵️‍♂️ CCTV ASSISTANT QUERY ===
Question: 'What is the color of the car the attackers use to run away?'
🦙 Asking Local LLaMA 3 to resolve context and rewrite the query...
🧠 Pruned Memory: NONE
✨ Refined Query: Who are the attackers and what do they use to run away?
🔢 Embedding refined natural language query...
🔍 Searching Vector Database for top matches...
🧩 System identified 1 distinct time event(s) across the video.
✂️ Extracting Clip 1/1: 0.0s to 90.8s...
📦 Loading Clip 1 inline to bypass Cloud Storage...
🧠 Asking Gemini 2.5 Flash to synthesize an answer...
🧹 Cleaning up local temporary files...

--- 🤖 FINAL ANSWER ---
The car the attackers use to run away is a **light-colored** sedan. Due to the black and white nature of the CCTV footage, it is not possible to determine the exact color, but it appears to be light, possibly silver or light grey.

The car is seen moving at **0:28** in the video, and the attackers get into it around **0:29**.


In [ ]:
# Point this to your completely raw, unannotated video
session_history = []
TARGET_VIDEO = "Anomaly-Videos-Part-1/Assault/Assault004_x264.mp4" # video with low resolution

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=45.0) ## do the ingestion overlapping

# --- STEP 2: CHAT ---
QUESTION = "after the attacker leave the screen, did anyone help the victim?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)


=== 🎬 STARTING DIRECT MULTIMODAL INGESTION: Assault004_x264 (107.4s) ===

⚙️ Processing Chunk 0 [0.0s - 45.0s]...
   ✅ Video chunk embedded and saved.

⚙️ Processing Chunk 1 [45.0s - 90.0s]...
   ✅ Video chunk embedded and saved.

⚙️ Processing Chunk 2 [90.0s - 107.4s]...
   ✅ Video chunk embedded and saved.
=== 🎉 DIRECT INGESTION COMPLETE! 'Assault004_x264' is now searchable. ===

=== 🕵️‍♂️ CCTV ASSISTANT QUERY ===
Question: 'after the attacker leave the screen, did anyone help the victim?'
🦙 Asking Local LLaMA 3 to resolve context and rewrite the query...
🧠 Pruned Memory: NONE
✨ Refined Query: After the attacker leaves the screen, was there any assistance provided to the victim?
🔢 Embedding refined natural language query...
🔍 Searching Vector Database for top matches...
🧩 System identified 1 distinct time event(s) across the video.
✂️ Extracting Clip 1/1: 0.0s to 110.4s...
📦 Loading Clip 1 inline to bypass Cloud Storage...
🧠 Asking Gemini 2.5 Flash to synthesize an answer...
🧹 Clean

In [42]:
QUESTION = "How did the attacker flee the scene?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)


=== 🕵️‍♂️ CCTV ASSISTANT QUERY ===
Question: 'How did the attacker flee the scene?'
🦙 Asking Local LLaMA 3 to resolve context and rewrite the query...
🧠 Pruned Memory: NONE
✨ Refined Query: Here's how the attacker fled the scene: after leaving the victims, one person remained on their motorcycle and didn't approach or assist them. The attacker then presumably escaped from the scene.
🔢 Embedding refined natural language query...
🔍 Searching Vector Database for top matches...
🧩 System identified 1 distinct time event(s) across the video.
✂️ Extracting Clip 1/1: 0.0s to 110.4s...
📦 Loading Clip 1 inline to bypass Cloud Storage...
🧠 Asking Gemini 2.5 Flash to synthesize an answer...
🧹 Cleaning up local temporary files...

--- 🤖 FINAL ANSWER ---
The attacker(s) fled the scene in a car.

*   **At 01:30 in the provided video clip**, the individuals involved in the attack begin to get back into the car that had been waiting with its headlights on.
*   **By 01:38**, the car starts to move away

In [43]:
QUESTION = "Give me the car's characteristic?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)


=== 🕵️‍♂️ CCTV ASSISTANT QUERY ===
Question: 'Give me the car's characteristic?'
🦙 Asking Local LLaMA 3 to resolve context and rewrite the query...
🧠 Pruned Memory: RELEVANT HISTORY
✨ Refined Query: The attacker fled the scene in a car. One person was present on a motorcycle throughout the incident.

Note: I only included the relevant information from the chat history that is necessary to understand the current query, which is about the car's characteristic.
🔢 Embedding refined natural language query...
🔍 Searching Vector Database for top matches...
🧩 System identified 1 distinct time event(s) across the video.
✂️ Extracting Clip 1/1: 0.0s to 110.4s...
📦 Loading Clip 1 inline to bypass Cloud Storage...
🧠 Asking Gemini 2.5 Flash to synthesize an answer...
🧹 Cleaning up local temporary files...

--- 🤖 FINAL ANSWER ---
The car appears in **Clip 1** starting at **00:15**.

**Car Characteristics:**
*   **Type:** Sedan, 4-door.
*   **Color:** Light-colored, possibly silver or light grey (du

In [45]:
# Point this to your completely raw, unannotated video
session_history = []
TARGET_VIDEO = "Anomaly-Videos-Part-1/Abuse/Abuse002_x264.mp4"

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
#ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=45.0) ## do the ingestion overlapping

# --- STEP 2: CHAT ---
QUESTION = "Did any accident happends on the road?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)


=== 🎬 STARTING DIRECT MULTIMODAL INGESTION: Abuse002_x264 (28.8s) ===

⚙️ Processing Chunk 0 [0.0s - 28.8s]...
   ✅ Video chunk embedded and saved.
=== 🎉 DIRECT INGESTION COMPLETE! 'Abuse002_x264' is now searchable. ===

=== 🕵️‍♂️ CCTV ASSISTANT QUERY ===
Question: 'Did any accident happends on the road?'
🦙 Asking Local LLaMA 3 to resolve context and rewrite the query...
🧠 Pruned Memory: NONE
✨ Refined Query: What accidents occurred on the road?
🔢 Embedding refined natural language query...
🔍 Searching Vector Database for top matches...
🧩 System identified 1 distinct time event(s) across the video.
✂️ Extracting Clip 1/1: 0.0s to 31.8s...
📦 Loading Clip 1 inline to bypass Cloud Storage...
🧠 Asking Gemini 2.5 Flash to synthesize an answer...
🧹 Cleaning up local temporary files...

--- 🤖 FINAL ANSWER ---
Yes, an accident happened on the road.

**Time:** 00:02
**Clip:** Clip 1
**Description:** At the 0:02 mark, a person riding a motorcycle on the right side of the road, near the pedestri

In [47]:
# Point this to your completely raw, unannotated video
session_history = []
TARGET_VIDEO = "Anomaly-Videos-Part-1/Abuse/Abuse002_x264.mp4"

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
#ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=45.0) ## do the ingestion overlapping

# --- STEP 2: CHAT ---
QUESTION = "summary what happen in the video?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)


=== 🕵️‍♂️ CCTV ASSISTANT QUERY ===
Question: 'summary what happen in the video?'
🦙 Asking Local LLaMA 3 to resolve context and rewrite the query...
🧠 Pruned Memory: NONE
✨ Refined Query: What happened in the video?
🔢 Embedding refined natural language query...
🔍 Searching Vector Database for top matches...
🧩 System identified 1 distinct time event(s) across the video.
✂️ Extracting Clip 1/1: 0.0s to 31.8s...
📦 Loading Clip 1 inline to bypass Cloud Storage...
🧠 Asking Gemini 2.5 Flash to synthesize an answer...
🧹 Cleaning up local temporary files...

--- 🤖 FINAL ANSWER ---
The video captures multiple vehicles stopping on a multi-lane road near a crosswalk, followed by several people exiting these vehicles and gathering in the street.

Here's a detailed summary of the events:
*   **0:00 - 0:06 (Clip 1)**: Traffic is moving. A white van stops near a crosswalk. A silver minivan stops behind it in the same lane. In the adjacent lane, a black sedan and another silver sedan stop. Two people 

In [48]:
session_history = []

# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Normal_Videos_for_Event_Recognition/Normal_Videos_576_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=30.0)

# --- STEP 2: CHAT ---
QUESTION = "Summary the video?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)


=== 🎬 STARTING DIRECT MULTIMODAL INGESTION: Normal_Videos_576_x264 (375.9s) ===

⚙️ Processing Chunk 0 [0.0s - 30.0s]...
   ✅ Video chunk embedded and saved.

⚙️ Processing Chunk 1 [30.0s - 60.0s]...
   ✅ Video chunk embedded and saved.

⚙️ Processing Chunk 2 [60.0s - 90.0s]...
   ✅ Video chunk embedded and saved.

⚙️ Processing Chunk 3 [90.0s - 120.0s]...
   ✅ Video chunk embedded and saved.

⚙️ Processing Chunk 4 [120.0s - 150.0s]...
   ✅ Video chunk embedded and saved.

⚙️ Processing Chunk 5 [150.0s - 180.0s]...
   ✅ Video chunk embedded and saved.

⚙️ Processing Chunk 6 [180.0s - 210.0s]...
   ✅ Video chunk embedded and saved.

⚙️ Processing Chunk 7 [210.0s - 240.0s]...
   ✅ Video chunk embedded and saved.

⚙️ Processing Chunk 8 [240.0s - 270.0s]...
   ✅ Video chunk embedded and saved.

⚙️ Processing Chunk 9 [270.0s - 300.0s]...
   ✅ Video chunk embedded and saved.

⚙️ Processing Chunk 10 [300.0s - 330.0s]...
   ✅ Video chunk embedded and saved.

⚙️ Processing Chunk 11 [330.0s - 3

In [49]:

session_history = []

# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Anomaly-Videos-Part-2/Burglary/Burglary039_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=30.0)

# --- STEP 2: CHAT ---
QUESTION = "are they allow to enter the property?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)


=== 🎬 STARTING DIRECT MULTIMODAL INGESTION: Burglary039_x264 (314.0s) ===

⚙️ Processing Chunk 0 [0.0s - 30.0s]...
   ✅ Video chunk embedded and saved.

⚙️ Processing Chunk 1 [30.0s - 60.0s]...
   ✅ Video chunk embedded and saved.

⚙️ Processing Chunk 2 [60.0s - 90.0s]...
   ✅ Video chunk embedded and saved.

⚙️ Processing Chunk 3 [90.0s - 120.0s]...
   ✅ Video chunk embedded and saved.

⚙️ Processing Chunk 4 [120.0s - 150.0s]...
   ✅ Video chunk embedded and saved.

⚙️ Processing Chunk 5 [150.0s - 180.0s]...
   ✅ Video chunk embedded and saved.

⚙️ Processing Chunk 6 [180.0s - 210.0s]...
   ✅ Video chunk embedded and saved.

⚙️ Processing Chunk 7 [210.0s - 240.0s]...
   ✅ Video chunk embedded and saved.

⚙️ Processing Chunk 8 [240.0s - 270.0s]...
   ✅ Video chunk embedded and saved.

⚙️ Processing Chunk 9 [270.0s - 300.0s]...
   ✅ Video chunk embedded and saved.

⚙️ Processing Chunk 10 [300.0s - 314.0s]...
   ✅ Video chunk embedded and saved.
=== 🎉 DIRECT INGESTION COMPLETE! 'Burglar

In [50]:
session_history = []

# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Anomaly-Videos-Part-2/Burglary/Burglary002_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=30.0)

# --- STEP 2: CHAT ---
QUESTION = "is he allow to enter the property?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)


=== 🎬 STARTING DIRECT MULTIMODAL INGESTION: Burglary002_x264 (102.1s) ===

⚙️ Processing Chunk 0 [0.0s - 30.0s]...
   ✅ Video chunk embedded and saved.

⚙️ Processing Chunk 1 [30.0s - 60.0s]...
   ✅ Video chunk embedded and saved.

⚙️ Processing Chunk 2 [60.0s - 90.0s]...
   ✅ Video chunk embedded and saved.

⚙️ Processing Chunk 3 [90.0s - 102.1s]...
   ✅ Video chunk embedded and saved.
=== 🎉 DIRECT INGESTION COMPLETE! 'Burglary002_x264' is now searchable. ===

=== 🕵️‍♂️ CCTV ASSISTANT QUERY ===
Question: 'is he allow to enter the property?'
🦙 Asking Local LLaMA 3 to resolve context and rewrite the query...
🧠 Pruned Memory: NONE
✨ Refined Query: Is the man in the red shirt allowed to enter the property?
🔢 Embedding refined natural language query...
🔍 Searching Vector Database for top matches...
🧩 System identified 1 distinct time event(s) across the video.
✂️ Extracting Clip 1/1: 0.0s to 105.1s...
📦 Loading Clip 1 inline to bypass Cloud Storage...
🧠 Asking Gemini 2.5 Flash to synthesi